Phase 6: Encoding categorical variables

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [2]:
BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/"
demo = pd.read_sas(BASE + "DEMO_L.xpt", format="xport")
cbc  = pd.read_sas(BASE + "CBC_L.xpt",  format="xport")
ghb  = pd.read_sas(BASE + "GHB_L.xpt",  format="xport")
df = demo.merge(cbc, on="SEQN").merge(ghb, on="SEQN")
print(df.shape)

(7199, 51)


In [3]:
df["DMDEDUC2"] = df["DMDEDUC2"].replace({7: np.nan, 9: np.nan})

df = df.dropna(subset=["LBXGH"])
y = (df["LBXGH"] >= 5.7).astype(int)

cbc_numeric_cols = [c for c in [
    "LBXWBCSI", "LBXLYPCT", "LBXMOPCT", "LBXNEPCT", "LBXEOPCT", "LBXBAPCT",
    "LBXRBCSI", "LBXHGB", "LBXHCT", "LBXMCVSI", "LBXMCHSI", "LBXMC",
    "LBXRDW", "LBXPLTSI", "LBXMPSI",
] if c in df.columns]

X = df[["RIAGENDR", "RIDRETH3", "DMDEDUC2"] + cbc_numeric_cols]
print(X.shape, y.mean())

(6715, 18) 0.3516008935219658


In [4]:
edu_order = [[1.0, 2.0, 3.0, 4.0, 5.0]]

def make_preprocessor(education="ordinal"):
    if education == "ordinal":
        edu = Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OrdinalEncoder(categories=edu_order,
                                   handle_unknown="use_encoded_value",
                                   unknown_value=-1)),
        ])
    else:
        edu = Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OneHotEncoder(handle_unknown="ignore")),
        ])

    return ColumnTransformer([
        ("sex",  Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                           ("enc", OneHotEncoder(drop="if_binary"))]), ["RIAGENDR"]),
        ("race", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                           ("enc", OneHotEncoder(handle_unknown="ignore"))]), ["RIDRETH3"]),
        ("edu",  edu, ["DMDEDUC2"]),
        ("num",  Pipeline([("imp", SimpleImputer(strategy="median")),
                           ("sc", StandardScaler())]), cbc_numeric_cols),
    ])

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for edu_mode in ["ordinal", "onehot"]:
    for name, clf in [("logreg", LogisticRegression(max_iter=1000)),
                      ("rf", RandomForestClassifier(n_estimators=200, random_state=42))]:
        pipe = Pipeline([("pre", make_preprocessor(edu_mode)), ("clf", clf)])
        scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
        print(f"{edu_mode:8s} {name:7s} AUC = {scores.mean():.3f} ± {scores.std():.3f}")

ordinal  logreg  AUC = 0.727 ± 0.009
ordinal  rf      AUC = 0.733 ± 0.012
onehot   logreg  AUC = 0.727 ± 0.013
onehot   rf      AUC = 0.732 ± 0.013


## Phase 6 justification: encoding of education (`DMDEDUC2`)

**Decision:** Education is encoded as an ordinal variable (1 to 5, from "less than 9th grade" to "college graduate or above"). Sex and race/ethnicity are one-hot encoded because they are nominal.

**Comparison.** We compared ordinal and one-hot encoding of education using 5-fold stratified cross-validation (ROC AUC), with all encoders fit inside the training fold only:

| Education encoding | Logistic regression | Random forest |
|---|---|---|
| Ordinal | 0.727 ± 0.009 | 0.733 ± 0.012 |
| One-hot | 0.727 ± 0.013 | 0.732 ± 0.013 |

**Interpretation.** The two encodings perform the same. The differences between them (0.000 for logistic regression and 0.001 for random forest) are far smaller than the fold-to-fold standard deviations (about 0.01), so there is no evidence that either encoding predicts better. Random forest also scores slightly higher than logistic regression (about 0.006), but this too is within the variability across folds.

**Rationale for ordinal.** Because predictive performance is equivalent, we chose on other grounds:
1. Education levels have a natural order, so a single ordinal column preserves that information.
2. It adds one feature instead of five, giving a simpler model with a single coefficient for education. This supports the interpretability goal for the eventual order/skip policy.
3. It avoids sparse indicator columns for small categories.

**Limitation.** Ordinal encoding assumes the gap between consecutive education levels is equal. One-hot makes no such assumption. Since one-hot gave no measurable gain here, we judge that this assumption does not cost us accuracy. If a later analysis reports education effects in detail, we can revisit this using one-hot.

**Other encoding decisions.**
- `RIAGENDR` (sex): `OneHotEncoder(drop="if_binary")`, so it becomes one 0/1 column.
- `RIDRETH3` (race/ethnicity): `OneHotEncoder(handle_unknown="ignore")`, since the categories are nominal with no order.
- Codes 7 (refused) and 9 (don't know) in `DMDEDUC2` were set to missing and imputed with the most frequent category inside each training fold.
- All encoders sit inside a scikit-learn `Pipeline`, so they are fit on the training fold only and an unseen category in held-out data does not break the pipeline.